## Projeto Final da Disciplina de Aprendizagem de Máquina

**Professor:**
Leandro Carlos de Souza

**Alunos:**
* Thiago Ney Evaristo Rodrigues
* Villeneve de Oliveira Soares

### NASA Airfoil Self-Noise Dataset

#### Context

NASA dataset obtained from a series of aerodynamic and acoustic tests of two and three-dimensional airfoil blade sections conducted in an anechoic wind tunnel. The data was obtained from [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/291/airfoil+self+noise).

#### Content

The NASA dataset comprises different size [NACA 0012](http://airfoiltools.com/airfoil/details?airfoil=n0012-il) airfoils at various wind tunnel speeds and angles of attack. The span of the airfoil and the observer position were the same in all of the experiments.

This problem has the following inputs:

1. Frequency, in Hertzs. 
2. Angle of attack, in degrees. 
3. Chord length, in meters.
4. Free-stream velocity, in meters per second. 
5. Suction side displacement thickness, in meters. 

The only output is:

6. Scaled sound pressure level, in decibels. 

### Imports

In [ ]:
import math
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    r2_score,
)

import xgboost as xgb

import tensorflow as tf
from tensorflow import keras

/home/thiag/anaconda3/envs/tf-gpu/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-11-16 11:28:13.842602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-16 11:28:13.911055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:

### Data Reading

In [ ]:
df = pd.read_csv(
    "../data/airfoil_self_noise.dat",
    names=[
        "frequency",
        "attack_angle",
        "chord_length",
        "free_stream_velocity",
        "displacement_thickness",  # suction_side_displacement_thickness
        "scaled_sound_pressure",
    ],
    delimiter="\t",
)

display(df)
display(df.describe())
display(df.info())

### Exploratory Data Analysis

In [ ]:
print(f"Duplicate data: {df.duplicated().sum()}")

In [ ]:
# Auxiliary Function


def sturges_bins(x):
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    n = len(x)
    if n <= 1:
        return 1, np.inf

    k = math.ceil(math.log2(n)) + 1
    data_range = x.max() - x.min()
    h = data_range / k if data_range > 0 else np.inf
    h = math.ceil(h)

    return k, h


# Plot of Histograms

fig, axes = plt.subplots(ncols=3, nrows=2, figsize=(9, 6), constrained_layout=True)
axes = axes.flatten()

for idx, col in enumerate(df.columns):
    if col != "type":
        data = df[col].values
        k, _ = sturges_bins(data)

        sns.histplot(data=data, bins=k, kde=True, ax=axes[idx])
        axes[idx].set_title(f"{col.replace("features/", "")}")
        axes[idx].set_xlabel("")
        axes[idx].set_ylabel("")

In [ ]:
# Correlation Matrix

corr_matrix = df.corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

### Data Division

In [ ]:
inputs = [
    "frequency",
    "attack_angle",
    "chord_length",
    "free_stream_velocity",
    "displacement_thickness",
]

scaler = StandardScaler()
df = scaler.fit_transform(df)

X = df[:, 0:5]
y = df[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

### Linear Regression

In [ ]:
poly = PolynomialFeatures(4)

X_train = poly.fit_transform(X_train)
X_test = poly.transform(X_test)

linear_model = LinearRegression().fit(X_train, y_train)

In [ ]:
r2_score = linear_model.score(X_test, y_test)

print(f"R2 Score: {}")